In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, count, when, round

bronze_table_name = "workspace.aml_fraud_project.transactions_bronze"
df = spark.table(bronze_table_name)

print(f"Total records in Bronze: {df.count()}")
display(df.limit(3))

In [0]:
df_filtered = df.filter(col("type").isin("TRANSFER", "CASH_OUT"))

df_features = df_filtered \
    .withColumn("errorBalanceOrig", round(col("newbalanceOrig") + col("amount") - col("oldbalanceOrg"), 2)) \
    .withColumn("errorBalanceDest", round(col("oldbalanceDest") + col("amount") - col("newbalanceDest"), 2))

print("Filtered Data with Balance Errors:")
display(df_features.select("type", "amount", "errorBalanceOrig", "errorBalanceDest", "isFraud").limit(5))

In [0]:
windowSpec = Window.partitionBy("nameOrig").orderBy("step")

df_features = df_features.withColumn("transaction_count_so_far", count("step").over(windowSpec))

print("Data with Transaction Velocity Feature:")
display(df_features.select("nameOrig", "step", "amount", "transaction_count_so_far", "isFraud").limit(5))

In [0]:
final_columns = [
    "step", "type", "amount", 
    "oldbalanceOrg", "newbalanceOrig", "errorBalanceOrig",
    "oldbalanceDest", "newbalanceDest", "errorBalanceDest",
    "transaction_count_so_far", "isFraud" 
]

silver_df = df_features.select(*final_columns)

silver_table_name = "workspace.aml_fraud_project.transactions_silver"

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(silver_table_name)

display(spark.table(silver_table_name).limit(5))